In [1]:
import os
os.environ['TORCHDYNAMO_VERBOSE'] = '1'

from torch.optim import Adam
from tqdm import trange, tqdm
import torch

device = torch.device('cuda:0')

# Flex modules

In [2]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.attention.flex_attention import create_block_mask, flex_attention

# very important to the compilation independent of any class instance
flex_attention_ = torch.compile(flex_attention, dynamic=True)

class Flextension(nn.Module):
    """
    Flex extension to replace self attention in transformer encoder layer
    """
    def __init__(self, in_proj_weight, in_proj_bias, out_proj_weight, out_proj_bias, block_mask, attn_heads):
        super().__init__()
        # weight and bias for linear layers
        self.in_proj_weight = in_proj_weight
        self.in_proj_bias = in_proj_bias
        self.out_proj_weight = out_proj_weight
        self.out_proj_bias = out_proj_bias
        # attributes
        self.attn_heads = attn_heads
        self.batch_first = True
        self._qkv_same_embed_dim = True

        # attributes for flex_attention
        self.kernel_options = {"BLOCK_M": 32, "BLOCK_N": 32, "BLOCK_M1": 16, "BLOCK_N1": 32, "BLOCK_M2": 32,
                               "BLOCK_N2": 16, }
        self.block_mask = block_mask

    def forward(self, x, x1, x2, attn_mask=None, key_padding_mask=None, need_weights=False, is_causal=False):
        q, k, v = F.linear(x, self.in_proj_weight, bias=self.in_proj_bias).chunk(3, -1)
        q_ = q.unflatten(-1, (self.attn_heads, -1)).transpose(2, 1)

        k_ = k.unflatten(-1, (self.attn_heads, -1)).transpose(2, 1)
        v_ = v.unflatten(-1, (self.attn_heads, -1)).transpose(2, 1)
        y_ = flex_attention_(q_, k_, v_, kernel_options=self.kernel_options, block_mask=self.block_mask)
        y = y_.transpose(2, 1).flatten(2, -1)
        y = F.linear(y, self.out_proj_weight, bias=self.out_proj_bias)
        return y


class FlexFormer(nn.TransformerEncoderLayer):
    """
    Inject flex attention into transformer encoder layer
    """
    def __init__(self, in_channel: int, nhead: int, dim_ff_scale: int = 4, dropout: float = 0.1, activation=F.leaky_relu, block_mask=None):
        super().__init__(d_model=in_channel, nhead=nhead, dim_feedforward=in_channel * dim_ff_scale, dropout=dropout,
                         activation=activation, layer_norm_eps=1e-5, batch_first=True, norm_first=True)
        self.self_attn = Flextension(self.self_attn.in_proj_weight, self.self_attn.in_proj_bias,
                                     self.self_attn.out_proj.weight, self.self_attn.out_proj.bias, block_mask, attn_heads=nhead)


class FlexBlock(nn.Module):
    """
    Flex block with multiple flex attention layers rebuilding the residual bock structure. Introduces a hyper-residual in addition to the residual already used in the transformer encoder layer.
    """
    def __init__(self, patch_size: torch.Tensor, kernel: int, in_channel: int, nhead: int, device: str, n_layers:int):
        """

        :param patch_size: H, W of patch
        :param kernel: kernel size
        :param in_channel: number of channels
        :param nhead: number of heads used in flex attention
        :param device: device to compile the block mask on (can be "cuda" or "cpu")
        :param n_layers: number of flex attention layers to use
        """
        super().__init__()
        print(n_layers, 'x', in_channel, patch_size.tolist())
        #assert (patch_size % 2 == 0).all(), "Patch size must be even"
        self.kernel = torch.tensor([kernel, kernel], device=device)
        self.patch_size = patch_size.to(device)
        S = patch_size.prod().item()
        block_mask = create_block_mask(self.compute_mask, None, nhead, S, S, device, _compile=True)

        self.layers = nn.ModuleList([FlexFormer(in_channel, nhead, block_mask=block_mask) for _ in range(n_layers)])
        self.use_id_mapping = n_layers > 1 # only use hyper-residual if more than one layer

    def forward(self, x):
        identity = x
        # todo pos_emb. Right now positional information is provided by convolution in FlexNetAvgPoolModel
        for layer in self.layers:
            x = layer(x)
        if self.use_id_mapping:
            x += identity
        return x

    def compute_mask(self, b, h, q_idx, kv_idx):
        # unravel index
        q_x = q_idx % self.patch_size[1]
        q_y = q_idx // self.patch_size[1]
        kv_x = kv_idx % self.patch_size[1]
        kv_y = kv_idx // self.patch_size[1]

        # compute mask
        is_valid_x = (q_x - kv_x).abs() <= self.kernel[0] // 2
        is_valid_y = (q_y - kv_y).abs() <= self.kernel[1] // 2
        is_valid = is_valid_x & is_valid_y
        return is_valid

## Test Flex modules with toy examples

In [3]:
patch_size = torch.tensor([16]*2)
input_channels = 128

model_block = FlexBlock(patch_size, 3, input_channels, 4, 'cuda', n_layers=4)

4 x 128 [16, 16]


### MSE example

In [4]:
from copy import deepcopy

x = torch.randn(8, patch_size.prod().item(), input_channels).to(device)
y = torch.randn_like(x)

model = deepcopy(model_block).to(device)
optimizer = Adam(model.parameters(), lr=1e-3)
for i in trange(5000, disable=True):
    optimizer.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        y_hat = model(x)
        loss = F.mse_loss(y_hat, y)
    loss.backward()
    optimizer.step()
    if i % 500 == 0:
        tqdm.write(f"iter: {i}, loss: {loss.item():.4f}")

iter: 0, loss: 5.4126
iter: 500, loss: 0.8291
iter: 1000, loss: 0.7151
iter: 1500, loss: 0.6718
iter: 2000, loss: 0.6516
iter: 2500, loss: 0.6324
iter: 3000, loss: 0.6143
iter: 3500, loss: 0.6016
iter: 4000, loss: 0.5896
iter: 4500, loss: 0.5857


### Classification example with cross entropy loss + linear classifier

In [6]:
y = torch.randint(0, 4, (8,)).to(device)

model = deepcopy(model_block).to(device)
classifier = nn.Linear(input_channels, 4, bias=False).to(device)
optimizer = Adam(list(classifier.parameters()) + list(model.parameters()), lr=1e-3)
for i in trange(5000, disable=True):
    optimizer.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        y_hat = model(x)
        y_hat = classifier(y_hat.mean(1))
        loss = F.cross_entropy(y_hat, y)
    loss.backward()
    optimizer.step()
    if i % 500 == 0:
        tqdm.write(f"iter: {i}, loss: {loss.item():.4f}, acc: {(y_hat.argmax(-1) == y).float().mean().item():.4f}")

iter: 0, loss: 1.3682, acc: 0.5000
iter: 500, loss: 0.0000, acc: 1.0000
iter: 1000, loss: 0.0000, acc: 1.0000
iter: 1500, loss: 0.0000, acc: 1.0000
iter: 2000, loss: 0.0000, acc: 1.0000
iter: 2500, loss: 0.0000, acc: 1.0000
iter: 3000, loss: 0.0000, acc: 1.0000
iter: 3500, loss: 0.0000, acc: 1.0000
iter: 4000, loss: 0.0000, acc: 1.0000
iter: 4500, loss: 0.0000, acc: 1.0000


# FlexNet

In [3]:
from typing import List


class PoolWithChannelExpansion(nn.Module):
    """
    Pooling layer that expands the number of channels via MLP or convolution
    """

    def __init__(self, patch_size: torch.Tensor, in_channel: int, out_channel: int, kernel_size: int, stride: int,
                 padding: int, pool_op: str = 'avg'):
        """

        :param patch_size: patch size
        :param in_channel: number of input channels
        :param out_channel: number of channels after pooling
        :param kernel_size: kernel size to use for pooling
        :param stride: pooling stride
        :param padding: pooling padding
        :param pool_op: operation to use for pooling
        """
        super().__init__()
        self.patch_size = patch_size
        self.unflatten = nn.Unflatten(2, patch_size.tolist())
        if pool_op == 'avg':
            self.pool = nn.Sequential(
                nn.AvgPool2d(kernel_size, stride, padding),
                nn.Conv2d(in_channel, out_channel, 1, 1, 0)
            )
        elif pool_op == 'max':
            self.pool = nn.Sequential(
                nn.MaxPool2d(kernel_size, stride, padding),
                nn.Conv2d(in_channel, out_channel, 1, 1, 0)
            )
        elif pool_op == 'conv_block': # non linear channel expansion
            self.pool = nn.Sequential(
                nn.Conv2d(in_channel, out_channel, kernel_size, stride, padding, bias=False, groups=in_channel),
                nn.InstanceNorm2d(out_channel, affine=True),
                nn.LeakyReLU()
            )
        elif pool_op == 'conv':
            self.pool = nn.Conv2d(in_channel, out_channel, kernel_size, stride, padding, bias=False, groups=in_channel)
        else:
            raise ValueError(f"Unknown pooling operation: {pool_op}")

    def forward(self, x):
        # x (B, H*W, C)
        x = x.permute(0, 2, 1)  # (B, C, H*W)
        x = self.unflatten(x)  # (B, C, H, W)
        y = self.pool(x)
        y_ = y.flatten(2, 3)  # (B, C', H*W)
        y_ = y_.permute(0, 2, 1)  # (B, H*W, C')
        return y_


class FlexNetAvgPoolModel(nn.Module):
    def __init__(self, n_channel: int, n_classes: int, n_heads: int = 4, group_repeats: List[int] = [2, 2, 2, 2],
                 patch_size: List[int] = [224, 224], pool_op: str = 'conv',
                 device: str = 'cuda'):
        """
        FlexNet model for classification with pooling and channel expansion for downsampling. Uses Flex Attention in in a residual block manner.
        :param n_channel: input channels
        :param n_classes: number of classes to classify
        :param n_heads: number of heads to use in flex attention
        :param group_repeats: Group repeats for the flex blocks. See table 1 in paper. Default is for ResNet-18
        :param patch_size: image size (H, W)
        :param pool_op: pooling operation to use. Since we have not implemented positional embedding in this version, should be 'conv' or 'conv_block'
        :param device: device to compile the block mask for flex attention on (can be "cuda" or "cpu")
        """
        super().__init__()
        patch_size = torch.tensor(patch_size)
        flexblock_kwargs = dict(nhead=n_heads, device=device)

        n_out_channels = 64
        self.first_layer = nn.Sequential(
            PoolWithChannelExpansion(patch_size.clone(), n_channel, n_out_channels, 7, 2, 3),
            FlexBlock(patch_size.clone() // 2, 3, n_out_channels, n_layers=1, **flexblock_kwargs),
        )
        patch_size //= 2

        self.layers = nn.ModuleList()
        for i, repeats in enumerate(group_repeats):
            if i == 0:  # max pool of second group
                self.layers.append(
                    PoolWithChannelExpansion(patch_size.clone(), n_out_channels, n_out_channels, 3, 2, 1, 'max'))
            else:
                self.layers.append(
                    PoolWithChannelExpansion(patch_size.clone(), n_out_channels, n_out_channels * 2, 3, 2, 1, pool_op))
                n_out_channels *= 2
            patch_size //= 2
            print(f'Group {i + 1}: {patch_size.tolist()} with {n_out_channels} channels')
            self.layers.append(
                FlexBlock(patch_size.clone(), 3, n_out_channels, n_layers=2 * repeats, **flexblock_kwargs),
            )
            self.classifier = nn.Linear(n_out_channels, n_classes)

    def forward(self, x):
        # view for transformer
        x_ = x.flatten(2).permute(0, 2, 1).contiguous()  # (B, C, H*W)
        x_ = self.first_layer(x_)
        for i, layer in enumerate(self.layers):
            x_ = layer(x_)
        # avg pooling
        x_ = x_.mean(1)
        y_hat = self.classifier(x_)
        return y_hat

## Test FlexNet with toy examples

In [4]:
y = torch.randint(0, 4, (8,)).to(device)

x = torch.randn(8, 1, 224, 224).to(device)

model = FlexNetAvgPoolModel(1, 4, patch_size=[224]*2, group_repeats=[2, 2, 2, 2]).to(device)
#model = torch.compile(model, dynamic=False)

1 x 64 [112, 112]
Group 1: [56, 56] with 64 channels
4 x 64 [56, 56]
Group 2: [28, 28] with 128 channels
4 x 128 [28, 28]
Group 3: [14, 14] with 256 channels
4 x 256 [14, 14]
Group 4: [7, 7] with 512 channels
4 x 512 [7, 7]


In [7]:
optimizer = Adam(model.parameters(), lr=1e-4)
for i in trange(1000, disable=True):
    optimizer.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        y_hat = model(x)
        loss = F.cross_entropy(y_hat, y)
    loss.backward()
    optimizer.step()
    if i % 50 == 0:
        tqdm.write(f"iter: {i}, loss: {loss.item():.4f}, acc: {(y_hat.argmax(-1) == y).float().mean().item():.4f}")

W0402 09:42:26.707000 3439567 site-packages/torch/_dynamo/convert_frame.py:985] [1/8] torch._dynamo hit config.recompile_limit (8)
W0402 09:42:26.707000 3439567 site-packages/torch/_dynamo/convert_frame.py:985] [1/8]    function: 'flex_attention' (/home/keuth/miniforge3/envs/FlexConvNightly/lib/python3.13/site-packages/torch/nn/attention/flex_attention.py:1161)
W0402 09:42:26.707000 3439567 site-packages/torch/_dynamo/convert_frame.py:985] [1/8]    last reason: 1/7: tensor 'key' size mismatch at index 3. expected 32, actual 64
W0402 09:42:26.707000 3439567 site-packages/torch/_dynamo/convert_frame.py:985] [1/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0402 09:42:26.707000 3439567 site-packages/torch/_dynamo/convert_frame.py:985] [1/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.


iter: 0, loss: 2.2267, acc: 0.3750
iter: 50, loss: 1.2637, acc: 0.3750
iter: 100, loss: 1.2144, acc: 0.3750
iter: 150, loss: 1.1064, acc: 0.5000
iter: 200, loss: 0.0057, acc: 1.0000
iter: 250, loss: 0.0020, acc: 1.0000
iter: 300, loss: 0.0008, acc: 1.0000
iter: 350, loss: 0.0004, acc: 1.0000
iter: 400, loss: 0.0004, acc: 1.0000
iter: 450, loss: 0.0004, acc: 1.0000
iter: 500, loss: 0.0002, acc: 1.0000
iter: 550, loss: 0.0002, acc: 1.0000
iter: 600, loss: 0.0001, acc: 1.0000
iter: 650, loss: 0.0001, acc: 1.0000
iter: 700, loss: 0.0001, acc: 1.0000
iter: 750, loss: 0.0001, acc: 1.0000
iter: 800, loss: 0.0001, acc: 1.0000
iter: 850, loss: 0.0001, acc: 1.0000
iter: 900, loss: 0.0001, acc: 1.0000
iter: 950, loss: 0.0001, acc: 1.0000


In [5]:
# check dynamic compilation with different batch sizes
print(model(torch.randn(8, 1, 224, 224).to(device)).shape)
print(model(torch.randn(2, 1, 224, 224).to(device)).shape)
print(model(torch.randn(6, 1, 224, 224).to(device)).shape)

torch.Size([8, 4])
torch.Size([2, 4])
torch.Size([6, 4])
